# SensorGuard: frozen CPU vs CUDA XGBoost benchmark

This notebook runs the same frozen XGBoost configuration on CPU and NVIDIA CUDA. It fits on the training split and measures model agreement on the validation split. It never evaluates the official test split.

Before running: choose **Runtime > Change runtime type > T4 GPU**. A completed `report.json` is the evidence; the notebook itself is not evidence that CUDA ran.

In [ ]:
!nvidia-smi
import os
assert os.path.exists('/dev/nvidia0'), 'Select a Colab GPU runtime before continuing.'

In [ ]:
!rm -rf /content/sensorguard-ml
!git clone --branch agent/add-evidence-verifier --single-branch https://github.com/mghadia1/sensorguard-ml.git /content/sensorguard-ml
%cd /content/sensorguard-ml
%pip install -q -e .
import xgboost as xgb
print('XGBoost', xgb.__version__)
print(xgb.build_info())

In [ ]:
!sensorguard download --destination data/raw
!sensorguard audit --data data/raw/ai4i2020.csv

In [ ]:
!sensorguard gpu-benchmark --data data/raw/ai4i2020.csv --out outputs/cuda-benchmark/report.json --random-state 42 --repeats 15

In [ ]:
!sensorguard verify-evidence --report outputs/cuda-benchmark/report.json

In [ ]:
import json
from pathlib import Path
report_path = Path('outputs/cuda-benchmark/report.json')
report = json.loads(report_path.read_text())
assert report['status'] == 'verified_cuda_run'
assert report['protocol']['official_test_evaluated'] is False
assert report['rows']['test_evaluated'] == 0
assert report['protocol']['repeats'] == 15
assert report['protocol']['warmup_fits_discarded_per_device'] == 1
report

## Save the evidence

Download `outputs/cuda-benchmark/report.json` from the Colab Files panel and save it locally as `docs/evidence/cuda-colab-t4-report-n15.json`. Do not overwrite the five-run evidence. Do not claim statistical significance unless the emitted p-value supports it.

In [ ]:
from google.colab import files
files.download('outputs/cuda-benchmark/report.json')